<a href="https://colab.research.google.com/github/192524158simats-gif/ITA0616--Machine-learning/blob/main/Lab%20Experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import csv

# Read training data from CSV file
with open('training_data.csv', 'r') as file:
    reader = list(csv.reader(file))

attributes = reader[0][:-1]
examples = reader[1:]

# Find possible values of each attribute
domains = [
    set(row[i] for row in examples)
    for i in range(len(attributes))
]

# Initialize Specific and General boundaries
n = len(attributes)
S = {tuple(['Ø'] * n)}
G = {tuple(['?'] * n)}

# Check whether hypothesis h1 is more general than h2
def more_general(h1, h2):
    return all(a == '?' or a == b or b == 'Ø'
               for a, b in zip(h1, h2))

# Check whether a hypothesis covers an example
def covers(h, example):
    return all(a == '?' or a == b
               for a, b in zip(h, example))

# Candidate-Elimination Algorithm
for row in examples:

    x = row[:-1]
    label = row[-1]

    if label == 'Yes':

        # Remove inconsistent hypotheses from G
        G = {g for g in G if covers(g, x)}

        new_S = set()

        for s in S:
            h = list(s)

            for i in range(n):
                if h[i] == 'Ø':
                    h[i] = x[i]
                elif h[i] != x[i]:
                    h[i] = '?'

            new_S.add(tuple(h))

        S = {
            s for s in new_S
            if any(more_general(g, s) for g in G)
        }

    else:

        # Remove hypotheses from G that cover negative example
        new_G = set()

        for g in G:
            if covers(g, x):

                for i in range(n):
                    if g[i] == '?':
                        for value in domains[i]:
                            if value != x[i] and value != 'Ø':
                                h = list(g)
                                h[i] = value
                                h = tuple(h)

                                if any(more_general(h, s) for s in S):
                                    new_G.add(h)
            else:
                new_G.add(g)

        G = new_G

        # Remove redundant hypotheses
        G = {
            g for g in G
            if not any(
                g != other and more_general(other, g)
                for other in G
            )
        }

    print("\nTraining Example:", row)
    print("Specific Boundary (S):", S)
    print("General Boundary (G):", G)

# Display final boundaries
print("\nFinal Specific Boundary:")
for h in S:
    print(h)

print("\nFinal General Boundary:")
for h in G:
    print(h)


Training Example: ['Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same', 'Yes']
Specific Boundary (S): {('Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same')}
General Boundary (G): {('?', '?', '?', '?', '?', '?')}

Training Example: ['Sunny', 'Warm', 'High', 'Strong', 'Warm', 'Same', 'Yes']
Specific Boundary (S): {('Sunny', 'Warm', '?', 'Strong', 'Warm', 'Same')}
General Boundary (G): {('?', '?', '?', '?', '?', '?')}

Training Example: ['Rainy', 'Cold', 'High', 'Strong', 'Warm', 'Change', 'No']
Specific Boundary (S): {('Sunny', 'Warm', '?', 'Strong', 'Warm', 'Same')}
General Boundary (G): {('Sunny', '?', '?', '?', '?', '?'), ('?', 'Warm', '?', '?', '?', '?'), ('?', '?', '?', '?', '?', 'Same')}

Training Example: ['Sunny', 'Warm', 'High', 'Strong', 'Cool', 'Change', 'Yes']
Specific Boundary (S): {('Sunny', 'Warm', '?', 'Strong', '?', '?')}
General Boundary (G): {('Sunny', '?', '?', '?', '?', '?'), ('?', 'Warm', '?', '?', '?', '?')}

Final Specific Boundary:
('Sunny', 'Warm', '?', 'Stron